# Números que faltan

Dada una lista de enteros, encontrar los que faltan entre el menor y el mayor. Ejemplo: en `[0, 2, 3, 6]` faltan `[1, 4, 5]`.

## 1. Versión sencilla y versión optimizada

La versión ingenua comprueba si cada número esperado está en la lista, lo que cuesta $O(n \cdot m)$. Optimizarla con un conjunto para que cada comprobación sea $O(1)$, y comparar tiempos.

In [1]:
import random
import timeit

def faltan_lento(lista):
    return [n for n in range(min(lista), max(lista) + 1) if n not in lista]

def faltan(lista):
    presentes = set(lista)
    return [n for n in range(min(lista), max(lista) + 1) if n not in presentes]

print(faltan([0, 2, 3, 6]))
datos = random.sample(range(20_000), 15_000)
assert faltan(datos) == faltan_lento(datos)
for f in (faltan_lento, faltan):
    print(f"{f.__name__:12} {timeit.timeit(lambda: f(datos), number=1):.4f} s")

[1, 4, 5]
faltan_lento 0.8593 s
faltan       0.0015 s


## 2. Un único número que falta

Si la lista contiene los números de 0 a $n$ salvo uno, encontrarlo en $O(n)$ y $O(1)$ memoria con la suma de Gauss.

In [2]:
def unico_que_falta(lista):
    n = len(lista)
    return n * (n + 1) // 2 - sum(lista)

print(unico_que_falta([3, 0, 1, 5, 2]))

4


## 3. Número que falta con XOR

Resolver el ejercicio anterior con XOR: como $a \oplus a = 0$ y $a \oplus 0 = a$, al combinar con XOR los números de $0$ a $n$ y los de la lista solo sobrevive el que falta. A diferencia de la suma, no crece, así que no desborda en lenguajes con enteros de tamaño fijo.

In [3]:
from functools import reduce
from operator import xor

def unico_que_falta_xor(lista):
    return reduce(xor, range(len(lista) + 1), 0) ^ reduce(xor, lista, 0)

random.seed(2)
for _ in range(500):
    n = random.randint(1, 50)
    falta = random.randint(0, n)
    datos = [x for x in range(n + 1) if x != falta]
    random.shuffle(datos)
    assert unico_que_falta_xor(datos) == unico_que_falta(datos) == falta
print(unico_que_falta_xor([3, 0, 1, 5, 2]))

4


## 4. Dos números que faltan

La lista contiene los números de 1 a $n$ salvo dos, $a$ y $b$. El XOR de todo da $a \oplus b \neq 0$, y su bit a 1 más bajo (`x & -x`) vale 1 en uno de ellos y 0 en el otro. Repartiendo los números en dos grupos según ese bit, cada grupo esconde un solo número que falta. Encontrarlos en $O(n)$ y $O(1)$ memoria y comprobarlo con conjuntos.

In [4]:
from itertools import chain

def dos_que_faltan(lista):
    n = len(lista) + 2
    todos = reduce(xor, range(1, n + 1), 0) ^ reduce(xor, lista, 0)
    bit = todos & -todos
    a = 0
    for x in chain(range(1, n + 1), lista):
        if x & bit:
            a ^= x
    return tuple(sorted((a, todos ^ a)))

for _ in range(500):
    n = random.randint(2, 60)
    a, b = random.sample(range(1, n + 1), 2)
    datos = [x for x in range(1, n + 1) if x not in (a, b)]
    random.shuffle(datos)
    assert dos_que_faltan(datos) == tuple(sorted(set(range(1, n + 1)) - set(datos)))
print(dos_que_faltan([1, 4, 2, 6, 7]))

(3, 5)


## 5. Número repetido y número que falta

En una lista con los números de 1 a $n$, uno aparece dos veces ($r$) y otro falta ($f$). Encontrar ambos con dos ecuaciones: la diferencia de sumas da $r - f$ y la de sumas de cuadrados $r^2 - f^2 = (r - f)(r + f)$. Comprobarlo con `Counter`.

In [5]:
from collections import Counter

def repetido_y_falta(lista):
    n = len(lista)
    resta = sum(lista) - n * (n + 1) // 2
    resta_cuadrados = sum(x * x for x in lista) - n * (n + 1) * (2 * n + 1) // 6
    suma = resta_cuadrados // resta
    return (suma + resta) // 2, (suma - resta) // 2

for _ in range(500):
    n = random.randint(2, 60)
    r, f = random.sample(range(1, n + 1), 2)
    datos = [r if x == f else x for x in range(1, n + 1)]
    random.shuffle(datos)
    repetido = next(x for x, c in Counter(datos).items() if c == 2)
    falta = (set(range(1, n + 1)) - set(datos)).pop()
    assert repetido_y_falta(datos) == (repetido, falta)
print(repetido_y_falta([3, 1, 2, 5, 3]))

(3, 4)
